# 01 Elementary Data Analysis

## 1. Data Inspection



### 1.1 Load Data

In [ ]:
import pandas as pd

df = pd.read_csv(
    'data.csv', # or url
    na_values = ['N/A', 'Unknown'],
    usecols = None,
    skiprows = None,
    index_col = None,
    sep = ',',
    parse_dates = None
)

- ```na_values``` replaces specified strings with NaN
- ```usecols``` retrieves only selected columns from file
- ```skiprows``` skips rows in dataframe (e.g. if first few lines are unnecessary)
- ```index_col``` uses specified column to index (if none assigns an integer)
- ```sep``` allows Pandas to handle multiple delimiters in file
- ```parse_dates``` converts date columns into datetime objects

**Create description table:**

|Column|Description|
|-|-|
| Column 1 | Description |
| Column 2 | Description |

### 1.2 Basic Inspection

In [ ]:
# Basic shape
print(f"rows={df.shape[0]}, columns={df.shape[1]}")

# First n=10 rows
df.head(n=10)

# basic description
df.info()

# Statistical summary of numeric columns
df.describe()

# Statistical summary of string-valued columns
df.describe(include="str")

# Categories and counts
df["categorical_column"].value_counts()

### 1.3 Train/Test Split
Safest to do this early on to avoid biasing our analysis.

In [ ]:
from sklearn.model_selection import train_test_split

target_col = "target_col"

X = df.drop(columns=target_col)
y = df[target_col]

X_train, y_train, X_test, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 50,
    shuffle = True,
    stratify = y
)

# OPTIONAL: create a training dataframe
train_df = X_train.copy()
train_df["target"] = y_train

- ```test_size``` defines the split, rule of thumb 80/20 train/test split
- ```random_state``` makes split repeatable
- ```shuffle``` shuffle the data before splitting
- ```stratify``` splits the training and test set evenly based on the chosen features or target

## 2. Data Quality



### 2.1 Missing value table


In [ ]:
# Create a new dataframe containing the sum of missing values, along with
# the percentage of missing values
missing_vals = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})
# Remove all rows without missing values, then sort according to the
# missing value percentage
missing_vals = missing_vals[missing_vals["Missing Count"] > 0].sort_values("Missing %", ascending=False)
# Print the final result
missing_vals

### 2.2 Missing value bar graph

In [ ]:
plt.subplots(figsize=(10, 4))
plt.barh(df.columns, df.isnull().sum())
for idx, val in enumerate(df.isnull().sum()):
    if val > 0:
        plt.text(val + 5, idx, str(val), va="center")
plt.title("# Missing Values")
plt.tight_layout()
plt.show()

**Make a note of what to drop and what to impute**

Drop row:
- Small fraction of rows missing
    - < 5% dropping probably fine
    - 5% - 10% investigate before dropping
    - \> 10% be careful
- Randomly distributed data (i.e. not concentrated on a single group)
- Not important enough to impute

Drop column:
- Large fraction of data is missing (>40% - 60%)
- Feature not important
- Unreliable reconstruction
- Redundant column

Numerical imputation:
- ```median``` is a safe default (especially for skewed data or containing outliers)
- ```mean``` when distribution is symmetric or has minimal outliers

Categorical imputation:
- ```most_frequent``` is a good default (i.e. mode)
- ```fillna("Missing")``` for when missingness contains information
- ```df["missing_indicator"]``` for when we want to track missing data, i.e. missingness may be useful for predicting target (good for tree-based models)

**NB:** Never calculate imputation values on the full dataset.

## 3. Univariate Analysis

In [ ]:
df_train.hist(bins=50, figsize=(20,15))
plt.show()

### Numerical Feature Distribution

A set of histograms for the chosen numerical features, denoted *num_cols*, along with the mean and median.


In [ ]:
# Pick out the numerical columns
num_cols = ["col_1", "col_2", ...] # Replace with chosen columns

fig, axes = plt.subplots(
    len(num_cols), 1,
    figsize=(12, 5 * len(num_cols)),
    squeeze=False
)

axes = axes.flatten()
for idx, col in enumerate(num_cols):
    axes[idx].hist(df[col].dropna(), bins=30)
    axes[idx].axvline(df[col].mean(), color="red", linestyle="--", label=f"Mean: {df[col].mean():.2f}")
    axes[idx].axvline(df[col].median(), color="green", linestyle="--", label=f"Median: {df[col].median():.2f}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Count")
    axes[idx].legend()

plt.suptitle("Numerical Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Box Plots for Numerical Features
A simple set of box plots to check for outliers in relevant numerical datasets

In [1]:
num_cols = ["col_1", "col_2", ...]

fig, axes = plt.subplots(
    2, 3,
    figsize=(12, 1.5 * len(num_cols)),
    squeeze=False
)

axes = axes.flatten()

for idx, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[idx])
    axes[idx].set_title(f"{col} — Box Plot")
    
plt.suptitle("Outlier Detection via Box Plots", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### Categorical Features
A simple set of bar graphs that visualize categorical features.

In [ ]:
# Pick out the categorical columns
cat_cols = ["col_1", "col_2", ...] # Replace with chosen columns

fig, axes = plt.subplots(
    len(num_cols), 1,
    figsize=(12, 5 * len(num_cols)),
    squeeze=False
)
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[idx].bar(counts.index.astype(str), counts.values)
    axes[idx].set_title(f"Distribution of {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Count")
    for j, v in enumerate(counts.values):
        axes[idx].text(j, v + 5, str(v), ha="center", fontweight="bold")

plt.suptitle("Categorical Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Bivariate Analysis (Relationships)

Assuming our target variable is binary, we can plot this variable against our various features to observe their relationship. We call our **target variable** by the name *target_var*, to be replaced by the correct name in the applied analysis. We also assume that it takes on values 0 or 1.

### Numerical Features
Plot the distribution of our chosen numerical feature for both binary classes of the target variable, overlaying them both.

In [ ]:
target_var = "target_var"
num_col = "col_1"

plt.hist(df[df[target_var] == 0][num_col].dropna(), bins=30, color="red", label="target = 0")
plt.hist(df[df[target_var] == 1][num_col].dropna(), bins=30, color="green", label="target = 1")
plt.title(f"Distribution by {num_col}")
plt.xlabel(f"num_col")
plt.ylabel("Count")
plt.legend()


### Categorical Features
Bar graphs of our binary target variable against a chosen set of categorical variables.

In [ ]:
target_var = "target_var"
# Pick out the categorical columns
cat_cols = ["col_1","col_2", ...] # Replace with chosen columns

fig, axes = plt.subplots(n, m, figsize=(14, 5)) # Note n x m must equal the number of columns

for idx, col in enumerate(cat_cols):
    df_temp = df.groupby(col)[target_var].mean()
    axes[idx].bar(df_temp.index, df_temp.values)
    axes[idx].set_title(f"{target_var} by {col}")
    axes[idx].set_ylabel(target_var)
    axes[idx].axhline(y=df[target_var].mean(), color="red", linestyle="--", label="Overall Rate")
    axes[idx].legend()
    
plt.suptitle("Categorical Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


### Two Categorical Features

In some cases, we may want to compare two categorical features on the one plot.

In [ ]:
target_var = "target_var"
cat_cols = ["col_1", "col_2"] # Note we group by col_1, then subgroup by col_2

df_grouped = df.groupby([cat_cols[0], cat_cols[1]])[target_var].mean().unstack()
df_grouped.plot(kind="bar", color=["red", "blue"])
plt.title(f"{target_var} by {cat_cols[0]} and {cat_cols[1]}")
plt.ylabel(f"{target_var}")
plt.xlabel(f"{cat_cols[0]}")
plt.legend(title=f"{cat_cols[1]}")
plt.tight_layout()
plt.xticks(rotation=0)
plt.show()

## Correlation Analysis

### Correlation Heatmap

In [ ]:
# First pick out our columns (these can be numeric or categorical)
num_cols = ["col_1", "col_2", ...]
numeric_df = df[num_cols].copy()
# Change binary columns to 0/1 numeric
numeric_df["bin_col"] = df["bin_col"].map({"val_1": 0, "val_2": 1})

corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Correlation With Target Variable

In [ ]:
target_var = "target_var"

# First pick out our columns (these can be numeric or categorical)
num_cols = ["col_1", "col_2", ...]
numeric_df = df[num_cols].copy()

# Change binary columns to 0/1 numeric
numeric_df["bin_col"] = df["bin_col"].map({"val_1": 0, "val_2": 1})

corr = numeric_df.corr()

# Correlation with target
target_corr = corr[target_var].drop(target_var).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.set_title(f"Correlation with {target_var}")
ax.set_xlabel("Correlation Coefficient")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 4. Target Analysis